# LangGraph Agent Bot with GLM (ZhipuAI)
A simple LangGraph agent using GLM API as the LLM.

In [1]:
# Install dependencies if needed
%pip install zhipuai langgraph langchain-core python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from typing import TypedDict, List
import os
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from zhipuai import ZhipuAI

load_dotenv()

api_key = os.getenv("ZHIPUAI_API_KEY")
if not api_key:
    raise ValueError("ZHIPUAI_API_KEY not found in environment variables")
print(f"API Key loaded: {api_key[-10:]}")

# Initialize the GLM client
client = ZhipuAI(api_key=api_key)

API Key loaded: C2ccAmlSIV


In [ ]:
# Test different GLM models to find one that works
test_models = ["glm-4-flash", "glm-3-turbo", "glm-4", "glm-4-plus", "glm-4-air", "chatglm3-6b"]

working_model = None
for model_name in test_models:
    try:
        print(f"Testing {model_name}...")
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": "Hello"}],
            temperature=0,
        )
        print(f"✅ {model_name} works! Response: {response.choices[0].message.content[:50]}")
        working_model = model_name
        break
    except Exception as e:
        print(f"❌ {model_name} failed: {str(e)[:100]}")

if working_model:
    print(f"\n🎉 Using model: {working_model}")
else:
    print("\n❌ No models worked. Check your API key and account permissions.")

# Create a wrapper function for GLM that matches LangChain's interface
def call_glm_api(messages: List[BaseMessage]) -> AIMessage:
    """Convert LangChain messages to GLM format and call the API"""
    # Convert LangChain messages to GLM format
    glm_messages = []
    for msg in messages:
        if isinstance(msg, HumanMessage):
            glm_messages.append({"role": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            glm_messages.append({"role": "assistant", "content": msg.content})

    # Use the working model we found
    if not working_model:
        raise ValueError("No working GLM model found. Please check your API key and permissions.")
    
    response = client.chat.completions.create(
        model=working_model,
        messages=glm_messages,
        temperature=0,
    )

    # Convert response back to LangChain format
    return AIMessage(content=response.choices[0].message.content)

# Initialize the LLM interface
llm = call_glm_api

In [6]:
# Define agent state
class AgentState(TypedDict):
    messages: List[BaseMessage]

# Define the process node
def process(state: AgentState) -> AgentState:
    response = llm(state["messages"])  # Call function directly instead of .invoke()
    print(f"\nAI: {response.content}")
    return {"messages": state["messages"] + [response]}

In [7]:
# Build the graph
graph = StateGraph(AgentState)
graph.add_node("process", process)
graph.add_edge(START, "process")
graph.add_edge("process", END)
app = graph.compile()

print("Graph compiled successfully!")

Graph compiled successfully!


In [8]:
# Run the agent
user_input = input("Enter your message: ")
result = app.invoke({"messages": [HumanMessage(content=user_input)]})
print("\n--- Final State ---")
for msg in result["messages"]:
    role = "Human" if isinstance(msg, HumanMessage) else "AI"
    print(f"{role}: {msg.content}")

APIRequestFailedError: Error code: 400, with error text {"error":{"code":"1211","message":"模型不存在，请检查模型代码。"}}

In [ ]:
# Multi-turn chat loop (optional)
conversation = []

print("Multi-turn chat — type 'quit' to exit\n")
while True:
    user_input = input("You: ")
    if user_input.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    conversation.append(HumanMessage(content=user_input))
    result = app.invoke({"messages": conversation})
    ai_response = result["messages"][-1]
    conversation.append(ai_response)
    print(f"AI: {ai_response.content}\n")